In [0]:
# BLOQUE 1 — Parámetros y destinos de la compuerta de calidad Bronze

import re
import uuid
from datetime import datetime, timezone

catalog = "workspace"
schema = "gtfs_bogota"
staging_volume_path = f"/Volumes/{catalog}/{schema}/staging_gtfs"

# El Workflow entregará estos dos valores desde la tarea Raw.
# Los valores por defecto permiten ejecutar y aprender el notebook manualmente.
dbutils.widgets.text("feed_version", "2025-10-28", "Versión GTFS")
dbutils.widgets.text(
    "snapshot_sha256",
    "07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952",
    "SHA-256 del snapshot",
)
feed_version = dbutils.widgets.get("feed_version").strip()
snapshot_sha256 = dbutils.widgets.get("snapshot_sha256").strip().lower()

if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", feed_version):
    raise ValueError("feed_version debe tener el formato AAAA-MM-DD.")
if not re.fullmatch(r"[0-9a-f]{64}", snapshot_sha256):
    raise ValueError("snapshot_sha256 debe ser un hash SHA-256 de 64 caracteres.")

quality_run_id = str(uuid.uuid4())
checked_at_utc = datetime.now(timezone.utc)
staging_snapshot_path = (
    f"{staging_volume_path}/gtfs_static/"
    f"feed_version={feed_version}/"
    f"sha256={snapshot_sha256}"
)
quality_audit_table = f"{catalog}.{schema}.audit_gtfs_bronze_snapshot_quality"
schema_observation_table = f"{catalog}.{schema}.audit_gtfs_source_schema_observations"

print(f"Corrida de calidad: {quality_run_id}")
print(f"Snapshot Bronze: versión={feed_version}, sha256={snapshot_sha256}")
print(f"Auditoría de calidad: {quality_audit_table}")


In [0]:
# BLOQUE 2 — Contrato verificable del snapshot GTFS

# Este contrato separa archivos obligatorios de los opcionales. Un archivo
# opcional ausente se documenta, pero no se convierte en una falla artificial.
gtfs_entities = [
    {"name": "agency", "file_name": "agency.txt", "table_name": "bronze_gtfs_agency", "required": True},
    {"name": "routes", "file_name": "routes.txt", "table_name": "bronze_gtfs_routes", "required": True},
    {"name": "stops", "file_name": "stops.txt", "table_name": "bronze_gtfs_stops", "required": True},
    {"name": "trips", "file_name": "trips.txt", "table_name": "bronze_gtfs_trips", "required": True},
    {"name": "calendar", "file_name": "calendar.txt", "table_name": "bronze_gtfs_calendar", "required": True},
    {"name": "calendar_dates", "file_name": "calendar_dates.txt", "table_name": "bronze_gtfs_calendar_dates", "required": True},
    {"name": "shapes", "file_name": "shapes.txt", "table_name": "bronze_gtfs_shapes", "required": True},
    {"name": "stop_times", "file_name": "stop_times.txt", "table_name": "bronze_gtfs_stop_times", "required": True},
    {"name": "frequencies", "file_name": "frequencies.txt", "table_name": "bronze_gtfs_frequencies", "required": False},
    {"name": "fare_attributes", "file_name": "fare_attributes.txt", "table_name": "bronze_gtfs_fare_attributes", "required": False},
    {"name": "fare_rules", "file_name": "fare_rules.txt", "table_name": "bronze_gtfs_fare_rules", "required": False},
    {"name": "feed_info", "file_name": "feed_info.txt", "table_name": "bronze_gtfs_feed_info", "required": False},
]

# Los identificadores críticos deben existir y no pueden estar vacíos.
identifier_contracts = {
    "routes": ["route_id"],
    "stops": ["stop_id"],
    "trips": ["trip_id", "route_id", "service_id"],
    "stop_times": ["trip_id", "stop_id"],
}

technical_columns = {
    "source_feed_version",
    "source_sha256",
    "source_file_path",
    "bronze_loaded_at_utc",
}


In [0]:
# BLOQUE 3 — Utilidades y esquema explícito de auditoría

import csv
import json

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    StructField,
    StructType,
    TimestampType,
)

quality_audit_schema = StructType([
    StructField("quality_run_id", StringType(), False),
    StructField("checked_at_utc", TimestampType(), False),
    StructField("source_feed_version", StringType(), False),
    StructField("source_sha256", StringType(), False),
    StructField("check_name", StringType(), False),
    StructField("check_category", StringType(), False),
    StructField("entity_name", StringType(), False),
    StructField("severity", StringType(), False),
    StructField("status", StringType(), False),
    StructField("observed_value", StringType(), False),
    StructField("expected_value", StringType(), False),
    StructField("details", StringType(), False),
])

schema_observation_schema = StructType([
    StructField("source_feed_version", StringType(), False),
    StructField("source_sha256", StringType(), False),
    StructField("source_file_name", StringType(), False),
    StructField("source_columns_json", StringType(), False),
    StructField("observed_at_utc", TimestampType(), False),
    StructField("quality_run_id", StringType(), False),
])

quality_rows = []

def add_check(
    check_name: str,
    check_category: str,
    entity_name: str,
    severity: str,
    status: str,
    observed_value: str,
    expected_value: str,
    details: str,
) -> None:
    """Acumula evidencia; la escritura ocurre una sola vez al final."""
    quality_rows.append((
        quality_run_id, checked_at_utc, feed_version, snapshot_sha256,
        check_name, check_category, entity_name, severity, status,
        str(observed_value), str(expected_value), details,
    ))


def read_gtfs_header(source_path: str) -> list[str]:
    """Lee solo el encabezado para observar cambios de esquema sin releer el archivo completo."""
    with open(source_path, "r", encoding="utf-8-sig", newline="") as source_file:
        header = next(csv.reader(source_file), None)

    if not header or any(not column.strip() for column in header):
        raise RuntimeError(f"Encabezado inválido o vacío en {source_path}.")
    if len(header) != len(set(header)):
        raise RuntimeError(f"El archivo tiene columnas duplicadas: {source_path}.")
    return [column.strip() for column in header]


def is_blank(column_name: str):
    """Considera inválidos tanto nulos como identificadores compuestos solo por espacios."""
    return F.col(column_name).isNull() | (F.trim(F.col(column_name)) == F.lit(""))


In [0]:
# BLOQUE 4 — Presencia, volumen y procedencia de cada entidad Bronze

# Validamos el archivo de Staging y su evidencia correspondiente en Bronze.
# Esta separación identifica si una falla ocurrió antes o después de la extracción.
try:
    available_staging_files = {item.name for item in dbutils.fs.ls(staging_snapshot_path)}
except Exception as error:
    raise RuntimeError(
        f"No se puede leer el snapshot de Staging: {staging_snapshot_path}"
    ) from error

snapshot_dfs = {}
entity_results = {}

for entity in gtfs_entities:
    entity_name = entity["name"]
    source_present = entity["file_name"] in available_staging_files
    table_name = f"{catalog}.{schema}.{entity['table_name']}"
    table_exists = spark.catalog.tableExists(table_name)

    if source_present:
        add_check(
            "source_file_present", "completeness", entity_name, "BLOCKING", "PASS",
            entity["file_name"], "archivo presente en Staging",
            "El archivo del snapshot fue extraído desde Raw.",
        )
    elif entity["required"]:
        add_check(
            "source_file_present", "completeness", entity_name, "BLOCKING", "FAIL",
            "ausente", "archivo GTFS obligatorio",
            f"No existe {entity['file_name']} en el snapshot de Staging.",
        )
    else:
        add_check(
            "source_file_present", "completeness", entity_name, "INFO", "NOT_APPLICABLE",
            "ausente", "archivo opcional",
            f"El proveedor no publicó {entity['file_name']} para esta versión.",
        )

    snapshot_df = None
    snapshot_count = 0
    if table_exists:
        snapshot_df = spark.table(table_name).filter(F.col("source_sha256") == snapshot_sha256)
        snapshot_count = snapshot_df.count()

    entity_results[entity_name] = {
        "source_present": source_present,
        "table_name": table_name,
        "table_exists": table_exists,
        "snapshot_count": snapshot_count,
    }

    if source_present and snapshot_count > 0:
        snapshot_dfs[entity_name] = snapshot_df
        add_check(
            "bronze_snapshot_rows", "completeness", entity_name, "BLOCKING", "PASS",
            snapshot_count, "> 0",
            f"Datos disponibles en {table_name}.",
        )

        wrong_version_count = snapshot_df.filter(
            F.col("source_feed_version") != F.lit(feed_version)
        ).count()
        add_check(
            "bronze_provenance_version", "lineage", entity_name, "BLOCKING",
            "PASS" if wrong_version_count == 0 else "FAIL",
            wrong_version_count, "0 filas con versión distinta",
            "El hash del snapshot debe corresponder a la versión declarada por el proveedor.",
        )
    elif source_present:
        add_check(
            "bronze_snapshot_rows", "completeness", entity_name, "BLOCKING", "FAIL",
            snapshot_count, "> 0",
            f"El archivo existe, pero no hay evidencia Bronze para {table_name}.",
        )
    else:
        add_check(
            "bronze_snapshot_rows", "completeness", entity_name, "INFO", "NOT_APPLICABLE",
            snapshot_count, "sin filas para archivo no publicado",
            "No se espera una carga Bronze si el proveedor omitió el archivo opcional.",
        )

display(
    spark.createDataFrame(quality_rows, quality_audit_schema)
    .filter(F.col("check_category").isin("completeness", "lineage"))
    .select("status", "severity", "check_name", "entity_name", "observed_value", "expected_value", "details")
    .orderBy("entity_name", "check_name")
)


In [0]:
# BLOQUE 5 — Identificadores y relaciones mínimas del modelo GTFS

# Bronze conserva la fuente; por eso una referencia rota se registra como advertencia
# para investigar al proveedor, sin reescribir ni descartar la evidencia original.
for entity_name, columns in identifier_contracts.items():
    snapshot_df = snapshot_dfs.get(entity_name)
    if snapshot_df is None:
        add_check(
            "critical_identifier_nulls", "validity", entity_name, "BLOCKING", "NOT_APPLICABLE",
            "sin datos Bronze", "identificadores disponibles",
            "No se evalúan identificadores cuando la entidad no superó la comprobación de presencia.",
        )
        continue

    missing_columns = sorted(set(columns) - set(snapshot_df.columns))
    if missing_columns:
        add_check(
            "critical_identifier_columns", "schema", entity_name, "BLOCKING", "FAIL",
            ", ".join(missing_columns), "columnas críticas presentes",
            "La tabla Bronze no conserva un identificador requerido por el contrato.",
        )
        continue

    null_metric_expressions = [
        F.sum(F.when(is_blank(column), F.lit(1)).otherwise(F.lit(0))).alias(column)
        for column in columns
    ]
    null_metrics = snapshot_df.agg(*null_metric_expressions).first().asDict()

    for column in columns:
        invalid_count = null_metrics[column]
        add_check(
            f"critical_identifier_{column}", "validity", entity_name, "BLOCKING",
            "PASS" if invalid_count == 0 else "FAIL",
            invalid_count, "0 nulos o vacíos",
            f"Validación de {column} para el snapshot completo.",
        )


def validate_relationship(
    relationship_name: str,
    child_entity: str,
    child_key: str,
    parent_entity: str,
    parent_key: str,
) -> None:
    """Reporta huérfanos sin convertir la fidelidad de Bronze en una corrección silenciosa."""
    child_df = snapshot_dfs.get(child_entity)
    parent_df = snapshot_dfs.get(parent_entity)

    if child_df is None or parent_df is None:
        add_check(
            relationship_name, "referential_integrity", child_entity, "WARNING", "NOT_APPLICABLE",
            "entidad no disponible", f"{child_entity}.{child_key} -> {parent_entity}.{parent_key}",
            "No se puede evaluar la relación hasta que ambas entidades estén cargadas.",
        )
        return

    child_keys = (
        child_df.filter(~is_blank(child_key))
        .select(F.col(child_key).alias("relationship_key"))
        .distinct()
    )
    parent_keys = (
        parent_df.filter(~is_blank(parent_key))
        .select(F.col(parent_key).alias("relationship_key"))
        .distinct()
    )
    orphan_count = child_keys.join(parent_keys, "relationship_key", "left_anti").count()

    add_check(
        relationship_name, "referential_integrity", child_entity, "WARNING",
        "PASS" if orphan_count == 0 else "WARN",
        orphan_count, "0 claves huérfanas distintas",
        f"{child_entity}.{child_key} debe existir en {parent_entity}.{parent_key}.",
    )


validate_relationship("trips_route_id_exists", "trips", "route_id", "routes", "route_id")
validate_relationship("stop_times_trip_id_exists", "stop_times", "trip_id", "trips", "trip_id")
validate_relationship("stop_times_stop_id_exists", "stop_times", "stop_id", "stops", "stop_id")

# Un service_id puede definirse en calendar.txt o exclusivamente como excepción
# en calendar_dates.txt; por eso la referencia válida es la unión de ambas fuentes.
trips_df = snapshot_dfs.get("trips")
calendar_df = snapshot_dfs.get("calendar")
calendar_dates_df = snapshot_dfs.get("calendar_dates")
if trips_df is not None and calendar_df is not None and calendar_dates_df is not None:
    declared_services = (
        calendar_df.filter(~is_blank("service_id")).select("service_id")
        .unionByName(calendar_dates_df.filter(~is_blank("service_id")).select("service_id"))
        .distinct()
        .withColumnRenamed("service_id", "relationship_key")
    )
    trip_services = (
        trips_df.filter(~is_blank("service_id"))
        .select(F.col("service_id").alias("relationship_key"))
        .distinct()
    )
    missing_service_count = trip_services.join(declared_services, "relationship_key", "left_anti").count()
    add_check(
        "trips_service_id_exists", "referential_integrity", "trips", "WARNING",
        "PASS" if missing_service_count == 0 else "WARN",
        missing_service_count, "0 servicios no declarados",
        "El servicio del viaje debe aparecer en calendar o calendar_dates.",
    )
else:
    add_check(
        "trips_service_id_exists", "referential_integrity", "trips", "WARNING", "NOT_APPLICABLE",
        "entidad no disponible", "calendar y calendar_dates disponibles",
        "No se pudo construir el conjunto de servicios declarados.",
    )


In [0]:
# BLOQUE 6 — Observación idempotente de evolución de esquema

# Cada combinación archivo + hash se registra una vez. En la siguiente versión,
# el Workflow podrá mostrar qué columnas aparecieron o dejaron de publicarse.
if not spark.catalog.tableExists(schema_observation_table):
    (
        spark.createDataFrame([], schema_observation_schema)
        .write
        .format("delta")
        .saveAsTable(schema_observation_table)
    )

schema_observation_rows = []
for entity in gtfs_entities:
    if not entity_results[entity["name"]]["source_present"]:
        continue

    source_path = f"{staging_snapshot_path}/{entity['file_name']}"
    current_columns = sorted(read_gtfs_header(source_path))

    previous_observation = (
        spark.table(schema_observation_table)
        .filter(F.col("source_file_name") == entity["file_name"])
        .filter(F.col("source_feed_version") < feed_version)
        .orderBy(F.col("source_feed_version").desc())
        .limit(1)
        .collect()
    )

    if previous_observation:
        previous_columns = set(json.loads(previous_observation[0]["source_columns_json"]))
        added_columns = sorted(set(current_columns) - previous_columns)
        removed_columns = sorted(previous_columns - set(current_columns))
        schema_status = "WARN" if added_columns or removed_columns else "PASS"
        schema_details = json.dumps({
            "compared_to_feed_version": previous_observation[0]["source_feed_version"],
            "added_columns": added_columns,
            "removed_columns": removed_columns,
        }, ensure_ascii=False)
        add_check(
            "source_schema_evolution", "schema", entity["name"], "WARNING", schema_status,
            json.dumps(current_columns, ensure_ascii=False), "sin cambios o revisión de cambios",
            schema_details,
        )
    else:
        add_check(
            "source_schema_evolution", "schema", entity["name"], "INFO", "NOT_APPLICABLE",
            json.dumps(current_columns, ensure_ascii=False), "línea base histórica",
            "No existe una observación anterior para comparar este archivo.",
        )

    current_observation_exists = (
        spark.table(schema_observation_table)
        .filter(F.col("source_sha256") == snapshot_sha256)
        .filter(F.col("source_file_name") == entity["file_name"])
        .limit(1)
        .count() > 0
    )
    if not current_observation_exists:
        schema_observation_rows.append((
            feed_version, snapshot_sha256, entity["file_name"],
            json.dumps(current_columns, ensure_ascii=False), checked_at_utc, quality_run_id,
        ))

if schema_observation_rows:
    (
        spark.createDataFrame(schema_observation_rows, schema_observation_schema)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(schema_observation_table)
    )

print(f"Observaciones de esquema creadas: {len(schema_observation_rows)}")


In [0]:
# BLOQUE 7 — Persistencia del resultado y decisión para el Workflow

# La auditoría conserva cada evaluación. Solo los fallos BLOCKING bloquean el Job;
# las advertencias quedan visibles para revisión antes de promover a Silver.
quality_df = spark.createDataFrame(quality_rows, quality_audit_schema)

if not spark.catalog.tableExists(quality_audit_table):
    (
        spark.createDataFrame([], quality_audit_schema)
        .write
        .format("delta")
        .saveAsTable(quality_audit_table)
    )

(
    quality_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(quality_audit_table)
)

summary_df = (
    quality_df
    .groupBy("status")
    .count()
    .orderBy("status")
)
display(summary_df)
display(quality_df.select("status", "severity", "check_name", "entity_name", "observed_value", "expected_value", "details").orderBy("status", "severity", "entity_name", "check_name"))

failed_error_count = quality_df.filter(
    (F.col("status") == "FAIL") & (F.col("severity") == "BLOCKING")
).count()
warning_count = quality_df.filter(F.col("status") == "WARN").count()
overall_status = "FAILED" if failed_error_count > 0 else ("WARNING" if warning_count > 0 else "PASSED")

print(f"Resultado del snapshot: {overall_status}")
print(f"Fallos bloqueantes: {failed_error_count}; advertencias: {warning_count}")

# Estas salidas permiten que el futuro Lakeflow Job comunique su resultado.
try:
    dbutils.jobs.taskValues.set(key="quality_status", value=overall_status)
    dbutils.jobs.taskValues.set(key="quality_run_id", value=quality_run_id)
except Exception as error:
    print(f"Task values no disponibles en esta ejecución interactiva: {error}")

if failed_error_count > 0:
    raise RuntimeError(
        f"La validación Bronze falló con {failed_error_count} error(es). "
        f"Consulta {quality_audit_table} para quality_run_id={quality_run_id}."
    )
    # Mostrar únicamente las advertencias de esta corrida de calidad.
display(
    quality_df
    .filter(F.col("status") == "WARN")
    .select(
        "check_name",
        "entity_name",
        "observed_value",
        "expected_value",
        "details",
    )
)
